The notebook extracts information from a corpus to build a knowledge graph.

It is a proof of concept of the pipeline and can be used as a demo.

In [1]:
from datetime import datetime
import hashlib
import json
from neo4j import GraphDatabase
import numpy as np
import ollama
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import sys
import time
from tqdm import tqdm
import torch
sys.path.insert(1, "src/graph/")
from graph_builder import (
    extract_graph,
    compute_chunk_embeddings,
    merge_graphs,
    validate_graph,
    build_neo4j_graph,
    feed_global_report,
    save_graph,
    load_to_neo4j,
    create_constraints
)
sys.path.insert(1, "src/preprocessing/")
from speeches import (
    load_speeches,
    split_into_chunks,
    load_prompt_template
)


# Goal

The goal of the pipeline is to extract information from a document. For this demonstration, we use a public speech as the dataset:

 ```src/test/2017-05-14_d-claration-de-m-emmanuel-macron-pr-sident-de-la-r-publique.txt```

The information extraction step is performed by an LLM. Several models are available, and their performance depends on several criteria:

* Model should be multilingual or specifically support French
* Can handle abstract concept 
* Can be executed on a home desktop in a efficient way

According to the litterature ```qwen2.5:7b``` model meet these criteria and provides good results for information extraction. We also use ```mistral:7b-instruct``` as comparison model.

# Install Ollama and the Qwen model

Before using the pipeline, install the following dependencies.

**Ollama** on Windows the install command is:

```
irm https://ollama.com/install.ps1 | iex
```

Check that the Ollama server is running:

```
ollama list
```

You can also check the API response directly:

```
curl http://localhost:11434/api/tags
```

If you've just installed Ollama, you won't have any models yet. Pull `qwen2.5:7b`:

```
ollama pull qwen2.5:7b
```

Check that it runs correctly:

```
ollama run qwen2.5:7b
```

Do the same thing with ```mistral:7b-instruct```

You can type anything into the prompt to confirm it responds.

# Load speech

We load the speech and made a short analysis about it

As you can see, the file is composed of several sections: title, date, source, speech, keywords, etc.

The speech itself is located between the following two sections:

* `Texte intégral`
* `MOTS CLÉS`

We will extract the speech while keeping the other available information. For this purpose, we will use a custom function.

The function returns a list of dictionaries. Since our dataset contains only one speech, we will store the dictionary in the **speech** variable.

Here the different keys field:

* id
* date
* title
* filename
* header
* text
* speakers
* chunks
* mots_cles
* circonstance
* intervenants

In [3]:
corpus = load_speeches(
    os.path.join(
        "src", "test", "speech")
    )
# take the last speech example
speech = corpus[-1]

Skipping benchmark_llm.txt: 'Texte intégral' marker not found


Let's take a look at the ```speakers```, ```intervenants```, ```circonstance``` and ```mot_cles```

In [4]:
print("speakers : ", speech["speakers"])
print("intervenants : ", speech["intervenants"])
print("circonstance : ", speech["circonstance"])
print("mots_cles : ", speech["mots_cles"])

speakers :  ['Emmanuel Macron']
intervenants :  ['Emmanuel Macron']
circonstance :  Voyage officiel au Royaume de Belgique, les 19 et 20 novembre 2018
mots_cles :  ['Relations internationales', 'Relations bilatérales France', 'France - Belgique', 'Relations bilatérales', 'Union européenne', 'Voyage']


In [5]:
print(speech["text"])

Sire, Majesté, les paroles que vous venez de prononcer sur la France nous touchent mon épouse et moi-même comme elles touchent le reste de ma délégation, parce que elles viennent de vous, parce que je sais que venant de vous, elles sont sincères, et parce qu'en vous, elles sont exprimées au nom du peuple belge pour lequel les Français éprouvent une fraternelle amitié. Permettez-moi, Majesté, de vous remercier au-delà de ces mots pour votre invitation à me rendre en Belgique en visite d'Etat, 47 ans après celle de Georges POMPIDOU.
J'ai vu dans cette invitation l'expression de qualités purement belges : l'audace, le refus des évidences, l'art de surprendre. Car quand il y a, comme entre la Belgique et la France, pays voisins que peu sépare, pas même un fleuve ou une montagne, une proximité que je qualifierais de familiale, il faut de l'audace pour provoquer la rencontre.
Soyez donc remercié, Sire, de m'offrir ainsi le temps de redécouvrir cette Belgique voisine et amie et de la comprend

# Embedding, informations and relations extraction

The current notebook is focused on informations extraction to feed a Neo4j graphdatabase for a graph RAG approch. 
It combines Retrieval-Augmented Generation (RAG), which connects an LLM to a document database, with a graph model that can capture relationships between entities.


In the previous notebook ```PoC_summarize.ipynb```, we explained how to summarize speeches and which models is better in french polictical domain.

However, if we want to extract information such as relationships, locations, subjects, and other entities, an encoder-decoder model is not sufficient. We can use a Large Language Model (LLM) to extract this information from the text.

In the current step, we set the chunk_overlap to 200 because we want to preserve contextual information between consecutive chunks.

The function ```split_into_chunks()``` call ```RecursiveCharacterTextSplitter``` from ```langchain_text_splitters``` do the chunk process

We will use the following approach:

* Split the speech on different chunks with overlap to keep context. 
* Analyze each chunk with a LLM to extract relevant information and relationships. 
* Generate an embedding for each chunk.

The embedding step transforms the text into a numerical vector representation. These vectors represent the text in a semantic vector space, making it possible to identify and retrieve similar or semantically related text. We will explain this process in more detail later.

In [6]:
#split the speech into chunks with overlap
speech["chunks"] = split_into_chunks(speech, chunk_size=1200, chunk_overlap=150)
print("Number of chunks : {0}".format(len(speech["chunks"])))

Number of chunks : 12


# Embedding

Chunk embedding transforms text into a numerical vector. It enables semantic search, allowing queries to be matched based on meaning and context rather than exact keywords. Combined with vector indexing, embeddings enable fast similarity searches to retrieve the most relevant context for an LLM prompt.

The vector lenght depends of the encoder model used. In the PoC we use ```bge-m3``` as it is multilingual and provides good results according to the current state of the art.

The function ```compute_chunk_embeddings``` performs the embedding step by calling the function ```SentenceTransformer``` from ```sentence_transformers```.

In [7]:
from concurrent.futures import ThreadPoolExecutor
embedding_executor = ThreadPoolExecutor(max_workers=4)
embedding_future = embedding_executor.submit(
                compute_chunk_embeddings,
                speech["chunks"]
            )

speech["chunks"] = embedding_future.result()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

**Each chunk has a embedded vector with a length of 1024**

In [8]:
print("length of the vector : ",len(speech["chunks"][0]['embedding']))

length of the vector :  1024


We can compare different vectors by computing their cosine similarity, which is the normalized dot product. It measures the angle between two vectors while ignoring their magnitude. Vectors pointing in the same direction have a high cosine similarity score, with a value close to 1.

To illustrate this, we can find the chunks related to the following topic: **"La relation entre la France et la belgique pour construire une Europe forte"**

NB: Before computing cosine similarity, make sure that the embedded vectors were generated using the same encoder model.

In [9]:
# First step  querry
query = "La relation entre la France et la belgique pour construire une Europe forte"
# Load the model
embedding_model = SentenceTransformer("BAAI/bge-m3")
# embedded the sentence
query_embeddings = embedding_model.encode(
    query,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=False
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [10]:
# extract all the chunks
matrix_chunks = np.array([chunk['embedding'] for chunk in speech["chunks"]])
# reshape to (1, dimensions)
query_2d = query_embeddings.reshape(1, -1)

# compute score
scores = cosine_similarity(query_2d, matrix_chunks).flatten()
# Display the ranking
print("Chunk ranking")
for index in np.argsort(scores)[::-1][:np.min([5, len(scores)])]:
    print("Chunk # : {0}, Score {1:.3f}".format(index, scores[index]))

# We can see the 2 best chunk
print("\nBest chunks")
for index in np.argsort(scores)[::-1][:2]:
    print(speech["chunks"][index]['text'])
    print("="*80)

print("\nWorst chunk")
print(speech["chunks"][np.argsort(scores)[0]]['text'])

Chunk ranking
Chunk # : 3, Score 0.692
Chunk # : 5, Score 0.683
Chunk # : 11, Score 0.668
Chunk # : 10, Score 0.643
Chunk # : 9, Score 0.630

Best chunks
Derrière cette proximité, il y a une histoire profonde. Depuis les ères romaines et carolingiennes, en passant par le brillant essor du comté de Flandre et de la civilisation bourguignonne, la Belgique s'est construite, vous l'avez évoqué, Majesté, à l'instant, comme un pont entre un monde germanique et latin, un carrefour où les grandes puissances européennes se sont bien souvent  il faut le dire  entrechoquées.
En France et en Belgique, nous avons traversé toutes les histoires d'Europe, et pendant des millénaires, cette histoire fut celle de fracas, de conflits, d'empires qui se déchirent, de rêves de domination, et nous avons, ensemble, commémoré ces derniers jours l'Armistice, et la fin de l'un de ces chocs les plus terribles.
Nous devons à ceux qui sont morts pour nous d'être aujourd'hui des femmes et des hommes de bonne volont

# Informations and relationship extraction

A Large Langage Model is used for information extraction steps. ```Qwen2.5:7b``` model was selected because it is multilingual, provides good results according to the literature, and is relatively lightweight. It can also be run locally on a desktop. 

For comparison, we also tested ```mistral:7b-instruct```, as the model was trained on French text.

For this PoC, we did not go further into the validation process. Instead, we focused on prompt engineering to optimize the extraction results as much as possible. This involved providing clear instructions and avoiding potential prompt poisoning by not including examples that could bias the model's output.

In [11]:
# folder with the prompt used to extract informations
prompt_template = load_prompt_template(
    os.path.join(
        "src",
        "prompts",
        "prompt_RAG_production_v4.txt"
    )
)
# Dict to save nodes and relationships
merged_graph = {
    "entities": [],
    "relations": []
}

# Let see the prompt use for the extraction step:

print(prompt_template)

Tu es un système d'extraction d'informations structurées pour un graphe de connaissances (GraphRAG).

## TÂCHE
Analyse UNIQUEMENT le texte entre <TEXT> et </TEXT>. Extrais les entités et relations explicitement présentes.

=== 1. ENTITÉS À EXTRAIRE ===

- Person : Uniquement les NOMS PROPRES de personnes physiques (ex: Prénom et/ou Nom de famille).
  * CONDITION : L'expression doit commencer par une majuscule et désigner nommément un individu.
  * EXCLUSION SYNTAXIQUE : Ne JAMAIS extraire un nom commun, un titre, un pronom ou un groupe nominal précédé d'un article ou d'un adjectif possessif ("le", "la", "mon", "notre", "votre").
- Country : États et pays souverains.
- City : Villes et communes uniquement.
- Organization : Entreprises, institutions publiques, administrations et alliances internationales.
- Event : Événements historiques ou officiels datés/nommés.
- Law : Traités, lois, constitutions, accords officiels.
- Theme : Subjectivités ou concepts politiques. RÈGLE : 1 à 3 mots m

In [12]:
# create prompts
prompts = [
    prompt_template.replace(
        "{text}", 
        chunk["text"]
        ) for chunk in speech["chunks"]
    ]
# extract relationship with llm_model
list_models = ["qwen2.5:7b", "mistral:7b-instruct"]
# list_models.append("hf.co/bartowski/Qwen2.5-14B-Instruct-GGUF:Qwen2.5-14B-Instruct-IQ3_XS")
graphs = {}
for llm_model in list_models:
    graphs[llm_model] = [
        extract_graph(
            prompts[i], 
            speech["chunks"][i]["text"], 
            model = llm_model
        ) for i in range(len(speech["chunks"]))
    ]

In [13]:
graphs["qwen2.5:7b"]

[{'entities': [{'type': 'Person', 'name': 'Sire'},
   {'type': 'Person', 'name': 'Majesté'},
   {'type': 'Person', 'name': 'Georges POMPIDOU'},
   {'type': 'Country', 'name': 'France'},
   {'type': 'Country', 'name': 'Belgique'}],
  'relations': []},
 {'entities': [{'type': 'City', 'name': 'Bruxelles'},
   {'type': 'Theme', 'name': 'compensation'},
   {'type': 'Theme', 'name': 'convergences'},
   {'type': 'Theme', 'name': 'différences'},
   {'type': 'Theme', 'name': 'nuances'}],
  'relations': []},
 {'entities': [{'type': 'Person', 'name': 'SIMENON'},
   {'type': 'Person', 'name': 'MAETERLINCK'},
   {'type': 'Person', 'name': 'VERHAEREN'}],
  'relations': []},
 {'entities': [{'type': 'Country', 'name': 'Belgique'},
   {'type': 'Country', 'name': 'France'},
   {'type': 'Theme', 'name': 'Armistice'},
   {'type': 'Theme', 'name': 'conflits'},
   {'type': 'Theme', 'name': 'empires qui se déchirent'},
   {'type': 'Theme', 'name': 'rêves de domination'}],
  'relations': []},
 {'entities': [{

In [14]:
for llm_model in list_models:
    print("\n\nResult with "+llm_model+" and prompt prompt_RAG_production_v4.2.txt")
    for i in range(len(speech["chunks"])):
        print("="*20+"Chunks ",i," "+"="*20)
        print(speech["chunks"][i]["text"])
        print(graphs[llm_model][i])



Result with qwen2.5:7b and prompt prompt_RAG_production_v4.2.txt
====================Chunks  0  ====================
Sire, Majesté, les paroles que vous venez de prononcer sur la France nous touchent mon épouse et moi-même comme elles touchent le reste de ma délégation, parce que elles viennent de vous, parce que je sais que venant de vous, elles sont sincères, et parce qu'en vous, elles sont exprimées au nom du peuple belge pour lequel les Français éprouvent une fraternelle amitié. Permettez-moi, Majesté, de vous remercier au-delà de ces mots pour votre invitation à me rendre en Belgique en visite d'Etat, 47 ans après celle de Georges POMPIDOU.
J'ai vu dans cette invitation l'expression de qualités purement belges : l'audace, le refus des évidences, l'art de surprendre. Car quand il y a, comme entre la Belgique et la France, pays voisins que peu sépare, pas même un fleuve ou une montagne, une proximité que je qualifierais de familiale, il faut de l'audace pour provoquer la rencontre

As you can see, both models have limitations and produce incorrect results. `Qwen2.5-7B` considers “Sire” and “Majesté” to be persons in Chunks 0 and 8. `Mistral-7B` considers “Herve” to be a person instead of a city. It also classifies “Monsieur le Bourgmestre” as an organization.

Below the results produced by the `Qwen2.5-14B`. The LLM has significantly more parameters and produces better results, as you can see (but it takes much more time on a middle config and cannot be used for the PoC).

# Curation pipeline

We can filter the results and remove some errors using regular expressions (regex) and other rule-based methods. By this way we keep only the relation we want (without waranty they are correct).
A fpythin function checks entitites and relations type to keep only those defined i`ALLOWED_ENTITY_TYPES` and `ALLOWED_RELATIONS`.